# BnSentMix mixed-script audit
This notebook downloads BnSentMix and counts rows that contain both Bengali-script characters and Latin letters in the same utterance. This mechanical gate is necessary but not sufficient; manually review samples. Candidate rows are not approved training data.

In [ ]:
!pip -q install pandas requests

In [ ]:
from pathlib import Path
import requests, pandas as pd, re, json

out = Path('/content/data/external/bnsentmix')
out.mkdir(parents=True, exist_ok=True)
url = 'https://huggingface.co/datasets/aplycaebous/BnSentMix/resolve/main/dataset.csv'
raw_path = out / 'bnsentmix_raw.csv'
r = requests.get(url, timeout=120)
r.raise_for_status()
raw_path.write_bytes(r.content)
df = pd.read_csv(raw_path, low_memory=False)
print('Rows:', len(df), 'Columns:', list(df.columns))
display(df.head(10))

In [ ]:
# Detect likely text column. Inspect this choice if the schema changes.
lower_cols = {str(c).lower().strip(): c for c in df.columns}
text_col = next((lower_cols[k] for k in ['text','sentence','utterance','comment','review','content'] if k in lower_cols), None)
if text_col is None:
    candidates = {c: (df[c].dropna().astype(str).str.len().gt(0).mean(), df[c].dropna().astype(str).str.len().median()) for c in df.columns}
    text_col = max(candidates, key=lambda c: candidates[c])
    print('Heuristic text column:', text_col, candidates[text_col])

audit_df = df.copy()
audit_df['source_row_id'] = range(len(audit_df))
audit_df['text'] = audit_df[text_col].fillna('').astype(str).str.strip()
audit_df['has_bengali_script'] = audit_df['text'].str.contains(r'[\u0980-\u09FF]', regex=True)
audit_df['has_latin_letters'] = audit_df['text'].str.contains(r'[A-Za-z]', regex=True)
audit_df['passes_two_script_gate'] = audit_df['has_bengali_script'] & audit_df['has_latin_letters']
print('Rows:', len(audit_df))
print('Non-empty:', int(audit_df['text'].str.len().gt(0).sum()))
print('Bengali script:', int(audit_df['has_bengali_script'].sum()))
print('Latin letters:', int(audit_df['has_latin_letters'].sum()))
print('Both scripts:', int(audit_df['passes_two_script_gate'].sum()))
display(audit_df.loc[audit_df['passes_two_script_gate'], ['source_row_id','text']].head(30))

In [ ]:
# Candidate pool only: rows passing both-script gate. Manual language review is still required.
mixed = audit_df.loc[audit_df['passes_two_script_gate'] & audit_df['text'].str.len().gt(0)].copy()
mixed['normalized_text'] = mixed['text'].str.casefold().str.replace(r'\s+', ' ', regex=True).str.strip()
dupes = int(mixed['normalized_text'].duplicated().sum())
mixed['label'] = 'bn-IN'
mixed['language_category'] = 'Bengali-English mixed-script'
mixed['source_dataset'] = 'BnSentMix'
mixed['source_url'] = 'https://huggingface.co/datasets/aplycaebous/BnSentMix'
mixed['review_status'] = 'pending_manual_language_and_quality_review'
mixed['license_status'] = 'verify_current_license_before_use'
candidate_path = out / 'bnsentmix_bengali_english_mixed_script_candidates.csv'
report_path = out / 'bnsentmix_mixed_script_audit.json'
mixed.drop(columns=['normalized_text'], errors='ignore').to_csv(candidate_path, index=False)
report = {'input_rows': int(len(df)), 'text_column': str(text_col), 'two_script_candidates': int(len(mixed)), 'unique_normalized_candidates': int(mixed['normalized_text'].nunique()), 'duplicate_rows': dupes, 'approved_rows': 0, 'warning': 'Two-script gate does not prove genuine code-mixing. Review actual language use and source license.'}
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Candidate:', candidate_path)
print('Audit:', report_path)

## Manual review checklist
- Does the sentence contain actual Bengali language content, not just a Bengali name or token?
- Is English genuinely used and written in Latin script?
- Is the utterance meaningful enough for a voice-agent language classifier?
- Is it duplicated or nearly duplicated?
- Does the source license permit the intended use?

If few or no rows pass, do not loosen the requirement just to hit 20,000. Find a more suitable source.